# PackSure — train the PDP segmentation model (YOLOv8n-seg, ONE class: `label_pdp`)

**Runtime → Change runtime type → T4 GPU** before running.

**Why one class?** `pdp_segmentation.py` takes the highest-confidence mask regardless of class. If you train a
second class (e.g. `barcode`) in the same model, a barcode mask can win and the "label" becomes a barcode.
Annotate the *printed label/PDP surface* as a **polygon** (Roboflow → *Smart Polygon*), class name `label_pdp`.
Barcodes are found by zxing-cpp on the raw frame and don't need a model yet.

**Data:** 400–600 real phone photos of Indian FMCG packs (pouches, boxes, curved bottles/cans, shelf clutter, angled shots,
some glare). Export from Roboflow as **YOLOv8** (segmentation) → zip → put at `MyDrive/packsure/dataset.zip`.

In [ ]:
!pip -q install ultralytics onnx onnxruntime
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch runtime to T4')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATASET_ZIP = '/content/drive/MyDrive/packsure/dataset.zip'
OUT_DIR     = '/content/drive/MyDrive/packsure/models'
!rm -rf /content/ds && mkdir -p /content/ds {OUT_DIR}
!unzip -q -o {DATASET_ZIP} -d /content/ds
!ls /content/ds

In [ ]:
import glob, os, yaml, collections
src = yaml.safe_load(open('/content/ds/data.yaml'))
print('exported yaml:', src)

# --- sanity checks that catch the mistakes that waste a training run -------------
bad_fmt, classes = 0, collections.Counter()
for f in glob.glob('/content/ds/*/labels/*.txt'):
    for line in open(f):
        p = line.split()
        if not p: continue
        classes[p[0]] += 1
        if len(p) <= 5: bad_fmt += 1          # a box (5 numbers), not a polygon
print('class ids seen:', dict(classes))
assert bad_fmt == 0, f'{bad_fmt} labels are bounding boxes, not polygons -> re-export a SEGMENTATION dataset'
assert set(classes) == {'0'}, 'Expected exactly one class (id 0 = label_pdp). Merge/remove other classes in Roboflow.'
for split in ('train','valid','test'):
    print(split, len(glob.glob(f'/content/ds/{split}/images/*')), 'images')

cfg = {'path': '/content/ds', 'train': 'train/images', 'val': 'valid/images', 'names': {0: 'label_pdp'}}
if os.path.isdir('/content/ds/test/images'): cfg['test'] = 'test/images'
yaml.safe_dump(cfg, open('/content/packaging.yaml', 'w'))
print(open('/content/packaging.yaml').read())

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n-seg.pt')
model.train(
    data='/content/packaging.yaml',
    epochs=80, patience=20, imgsz=640, batch=16, device=0,
    # augmentation tuned for phone photos of packs
    degrees=12, perspective=0.0008, scale=0.4, translate=0.1,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    fliplr=0.0,          # labels are never mirrored in real use
    mosaic=1.0, close_mosaic=10,
    project='/content/runs', name='pdp_seg', exist_ok=True,
)

In [ ]:
best = YOLO('/content/runs/pdp_seg/weights/best.pt')
m = best.val(data='/content/packaging.yaml')
print('mask mAP50   :', round(m.seg.map50, 3))
print('mask mAP50-95:', round(m.seg.map, 3))
# Rough guide: mask mAP50 >= 0.85 is good enough to beat the contour fallback on real shelves.
# If it is < 0.7, you almost certainly need more (and more varied) photos, not more epochs.

In [ ]:
import shutil
onnx_path = best.export(format='onnx', imgsz=640, opset=12, simplify=True)
shutil.copy('/content/runs/pdp_seg/weights/best.pt', f'{OUT_DIR}/pdp_yolov8n_seg.pt')
shutil.copy(onnx_path,                                f'{OUT_DIR}/pdp_yolov8n_seg.onnx')
print('Saved to', OUT_DIR)
!ls -la {OUT_DIR}

In [ ]:
# Verify the ONNX file loads through the SAME call pdp_segmentation.py uses, and returns masks.
import cv2, glob, numpy as np
from ultralytics import YOLO
seg = YOLO(f'{OUT_DIR}/pdp_yolov8n_seg.onnx')
for p in glob.glob('/content/ds/test/images/*')[:5] or glob.glob('/content/ds/valid/images/*')[:5]:
    img = cv2.imread(p); r = seg.predict(img, verbose=False)[0]
    ok = r.masks is not None and len(r.masks.data) > 0
    print(os.path.basename(p), 'mask OK' if ok else 'NO MASK', '' if not ok else f'conf={float(r.boxes.conf.max()):.2f}')
# If .onnx misbehaves, delete it from backend/models/ and keep only the .pt -- the loader falls back to it.

## Deploy
Copy both files into `backend/models/` (they are git-ignored). Restart the backend: `/health` should report
`pdp: yolov8n-seg loaded`, and scans will report `pdp_detection_method = yolov8n_seg`.